# Pathfinding SNCF - Dijkstra sur le reseau ferroviaire

Ce notebook construit un graphe du reseau ferroviaire francais a partir des donnees GTFS SNCF,
puis implemente l'algorithme de Dijkstra pour trouver le trajet optimal entre deux villes.

## Sommaire
1. **Exploration des donnees GTFS** - Comprendre la structure
2. **Parsing et nettoyage** - Charger stops, trips, stop_times
3. **Construction du graphe ferroviaire** - Aretes reelles entre gares
4. **Algorithme de Dijkstra** - Plus court chemin en temps de trajet
5. **Recherche ville -> gare** - Matching intelligent
6. **Tests et resultats** - Exemples concrets

---
## Etape 1 : Exploration des donnees GTFS

Le format GTFS (General Transit Feed Specification) est le standard mondial pour les donnees de transport en commun.

### Structure des fichiers disponibles :
| Fichier | Description | Lignes |
|---------|-------------|--------|
| `stops.txt` | Gares avec coordonnees GPS | ~8 900 |
| `routes.txt` | Lignes ferroviaires (TGV, TER...) | ~840 |
| `trips.txt` | Trains specifiques sur une ligne | ~46 900 |
| `stop_times.txt` | Horaires d'arret a chaque gare | ~397 900 |

### Relations entre fichiers :
```
routes --(route_id)--> trips --(trip_id)--> stop_times --(stop_id)--> stops
```

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
import heapq
import time
import os
import warnings
warnings.filterwarnings('ignore')

if os.path.exists("../../../dataset/sncf/stops.txt"):
    GTFS_DIR = "../../../dataset/sncf/"
else:
    GTFS_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), "dataset", "sncf") + "/"

print(f"Repertoire GTFS : {GTFS_DIR}")
print(f"Fichiers disponibles : {os.listdir(GTFS_DIR)}")

Repertoire GTFS : /Users/atressel/Documents/Epitech/Projets/T-AIA-911-NAN_1/model/ALEXANDRE/dataset/sncf/


FileNotFoundError: [Errno 2] No such file or directory: '/Users/atressel/Documents/Epitech/Projets/T-AIA-911-NAN_1/model/ALEXANDRE/dataset/sncf/'

In [ ]:
# Chargement des fichiers GTFS
stops = pd.read_csv(f"{GTFS_DIR}stops.txt")
routes = pd.read_csv(f"{GTFS_DIR}routes.txt")
trips = pd.read_csv(f"{GTFS_DIR}trips.txt")
stop_times = pd.read_csv(f"{GTFS_DIR}stop_times.txt", low_memory=False)

print(f"stops:      {stops.shape[0]:>10,} lignes, {stops.shape[1]} colonnes")
print(f"routes:     {routes.shape[0]:>10,} lignes, {routes.shape[1]} colonnes")
print(f"trips:      {trips.shape[0]:>10,} lignes, {trips.shape[1]} colonnes")
print(f"stop_times: {stop_times.shape[0]:>10,} lignes, {stop_times.shape[1]} colonnes")

In [ ]:
# Exploration des gares (stops)
print("=== STOPS ===")
print(f"Colonnes: {list(stops.columns)}")
print(f"\nlocation_type:")
print(f"  1 = StopArea (gare parente)  : {(stops['location_type'] == 1).sum()}")
print(f"  0 = StopPoint (quai/voie)    : {(stops['location_type'] == 0).sum()}")
print(f"\nExemples de gares (StopArea) :")
stops[stops['location_type'] == 1][['stop_id', 'stop_name', 'stop_lat', 'stop_lon']].head(10)

In [ ]:
# Exploration des stop_times - c'est le fichier CLE pour le pathfinding
print("=== STOP_TIMES ===")
print(f"Colonnes: {list(stop_times.columns)}")
print(f"\nNombre de trips distincts: {stop_times['trip_id'].nunique():,}")
print(f"Nombre de stops distincts: {stop_times['stop_id'].nunique():,}")

# Distribution des arrets par trip
trip_sizes = stop_times.groupby('trip_id').size()
print(f"\n=== DISTRIBUTION DES ARRETS PAR TRIP ===")
print(trip_sizes.describe())
print(f"\nTrips avec 2 arrets (navettes) : {(trip_sizes == 2).sum():,}")
print(f"Trips avec 3-10 arrets          : {((trip_sizes >= 3) & (trip_sizes <= 10)).sum():,}")
print(f"Trips avec 10+ arrets            : {(trip_sizes > 10).sum():,}")
print(f"Max arrets dans un trip          : {trip_sizes.max()}")

In [ ]:
# Exemple d'un trip complet
example_trip = trip_sizes[trip_sizes > 5].index[0]
example = stop_times[stop_times['trip_id'] == example_trip].sort_values('stop_sequence')
print(f"Exemple - arrets d'un trip :")
print(f"Trip ID: {example_trip[:80]}...")
example[['stop_id', 'arrival_time', 'departure_time', 'stop_sequence']]

---
## Etape 2 : Parsing et nettoyage des donnees

### Logique cle :
- Les `StopPoint` (location_type=0) sont les arrets reels dans `stop_times`
- Chaque `StopPoint` a un `parent_station` qui est un `StopArea` (la gare)
- On va travailler au niveau **StopArea** (gare) pour simplifier le graphe
- Le nom de la gare vient du `StopArea` parent

In [ ]:
# Creer les mappings StopPoint -> StopArea
stop_points = stops[stops['location_type'] == 0][['stop_id', 'stop_name', 'parent_station']].copy()
stop_areas = stops[stops['location_type'] == 1][['stop_id', 'stop_name', 'stop_lat', 'stop_lon']].copy()

# Dictionnaires de mapping
stoppoint_to_area = dict(zip(stop_points['stop_id'], stop_points['parent_station']))
area_id_to_name = dict(zip(stop_areas['stop_id'], stop_areas['stop_name']))
area_id_to_lat = dict(zip(stop_areas['stop_id'], stop_areas['stop_lat']))
area_id_to_lon = dict(zip(stop_areas['stop_id'], stop_areas['stop_lon']))

print(f"StopPoints (quais) : {len(stop_points):,}")
print(f"StopAreas  (gares) : {len(stop_areas):,}")
print(f"\nExemples de mapping StopPoint -> StopArea :")
for sp, sa in list(stoppoint_to_area.items())[:5]:
    name = area_id_to_name.get(sa, '?')
    print(f"  {sp[:50]:50s} -> {name}")

In [ ]:
def time_to_minutes(time_str):
    """Convertit un horaire HH:MM:SS en minutes depuis minuit.
    Gere les heures > 24 (trains de nuit, ex: 25:30:00 = 1h30 le lendemain)."""
    if pd.isna(time_str):
        return np.nan
    parts = str(time_str).strip().split(':')
    try:
        h = int(parts[0])
        m = int(parts[1]) if len(parts) > 1 else 0
        s = int(parts[2]) if len(parts) > 2 else 0
        return h * 60 + m + s / 60.0
    except (ValueError, IndexError):
        return np.nan

# Test
print(f"08:30:00 -> {time_to_minutes('08:30:00')} min")
print(f"14:15:30 -> {time_to_minutes('14:15:30'):.1f} min")
print(f"25:30:00 -> {time_to_minutes('25:30:00')} min (train de nuit, = 1h30 J+1)")

In [ ]:
# Enrichir stop_times avec le parent_station et les horaires en minutes
print("Enrichissement des stop_times...")
t0 = time.time()

stop_times['parent_station'] = stop_times['stop_id'].map(stoppoint_to_area)
stop_times['station_name'] = stop_times['parent_station'].map(area_id_to_name)
stop_times['arrival_min'] = stop_times['arrival_time'].apply(time_to_minutes)
stop_times['departure_min'] = stop_times['departure_time'].apply(time_to_minutes)

# Supprimer les lignes sans parent_station ou sans horaire
n_before = len(stop_times)
stop_times = stop_times.dropna(subset=['parent_station', 'arrival_min', 'departure_min'])
n_after = len(stop_times)

print(f"Fait en {time.time()-t0:.1f}s")
print(f"Lignes conservees: {n_after:,} / {n_before:,} ({n_before - n_after} supprimees)")
print(f"\nExemple :")
stop_times[['trip_id', 'station_name', 'arrival_time', 'departure_time', 'stop_sequence']].head(5)

In [ ]:
import re

def extract_train_type(stop_id):
    """Extrait le type de train depuis un stop_id.
    Ex: 'StopPoint:OCETrain TER-87342014' -> 'TER'
        'StopPoint:OCETGV INOUI-71043075' -> 'TGV INOUI'
        'StopPoint:OCENavette-87571000'   -> 'Navette'
    """
    m = re.match(r'StopPoint:OCE(.+)-\d+', str(stop_id))
    if m:
        raw = m.group(1).strip()
        if raw.startswith("Train "):
            return raw[6:]
        return raw
    return "Inconnu"

# Extraire le type de train pour chaque ligne de stop_times
stop_times['train_type'] = stop_times['stop_id'].apply(extract_train_type)

# Afficher les types de trains trouves
type_counts = stop_times['train_type'].value_counts()
print("=== TYPES DE TRAINS DANS LE RESEAU ===")
print(f"{'Type':<25} {'Nb arrets':>10} {'%':>7}")
print("-" * 44)
for t, c in type_counts.items():
    print(f"{t:<25} {c:>10,} {c/len(stop_times)*100:>6.1f}%")
print(f"\n{'TOTAL':<25} {len(stop_times):>10,}")

---
## Etape 3 : Construction du graphe ferroviaire

### Principe :
Pour chaque **trip** (train), les arrets consecutifs dans `stop_times` definissent une **arete** du graphe.

```
Trip TGV 6201: Paris-Gare-de-Lyon (08:00) -> Macon (09:30) -> Lyon-Part-Dieu (10:00)
                    |                              |                   |
                    +--- arete 1: 90 min ----------+                   |
                                                   +--- arete 2: 30 min ---+
```

On garde le **temps de trajet minimum** entre chaque paire de gares (sur l'ensemble des trips).

### Approche vectorisee (rapide) :
On utilise `shift()` de pandas au lieu d'iterer ligne par ligne.

In [ ]:
print("Construction du graphe ferroviaire (approche vectorisee)...")
t0 = time.time()

# Trier par trip et sequence
st = stop_times.sort_values(['trip_id', 'stop_sequence']).copy()

# Pour chaque ligne, regarder la ligne precedente (meme trip)
st['prev_trip'] = st['trip_id'].shift(1)
st['prev_station'] = st['parent_station'].shift(1)
st['prev_departure'] = st['departure_min'].shift(1)

# Garder seulement les lignes ou le trip est le meme que le precedent
# (= arrets consecutifs du meme train)
consecutive = st[st['trip_id'] == st['prev_trip']].copy()
consecutive['travel_time'] = consecutive['arrival_min'] - consecutive['prev_departure']

# Filtrer les temps de trajet positifs
consecutive = consecutive[consecutive['travel_time'] > 0]

print(f"Liaisons brutes (avant dedup) : {len(consecutive):,}")

# Agreger SANS filtre de type : garder le temps minimum et compter par liaison
edge_stats = consecutive.groupby(['prev_station', 'parent_station']).agg(
    min_time=('travel_time', 'min'),
    mean_time=('travel_time', 'mean'),
    nb_trains=('travel_time', 'count')
).reset_index()
edge_stats.columns = ['station_from', 'station_to', 'min_time', 'mean_time', 'nb_trains']

# Agreger PAR TYPE DE TRAIN : pour chaque liaison + type, garder le min et les types
edge_stats_typed = consecutive.groupby(['prev_station', 'parent_station', 'train_type']).agg(
    min_time=('travel_time', 'min'),
    mean_time=('travel_time', 'mean'),
    nb_trains=('travel_time', 'count')
).reset_index()
edge_stats_typed.columns = ['station_from', 'station_to', 'train_type', 'min_time', 'mean_time', 'nb_trains']

# Pour chaque liaison, lister les types de trains qui la desservent
edge_train_types = consecutive.groupby(['prev_station', 'parent_station'])['train_type'].apply(
    lambda x: sorted(set(x))
).reset_index()
edge_train_types.columns = ['station_from', 'station_to', 'train_types']
# Fusionner avec edge_stats
edge_stats = edge_stats.merge(edge_train_types, on=['station_from', 'station_to'], how='left')

print(f"\nFait en {time.time()-t0:.1f}s")
print(f"Aretes uniques (liaisons directes) : {len(edge_stats):,}")
print(f"Aretes par type de train            : {len(edge_stats_typed):,}")
print(f"Gares connectees : {len(set(edge_stats['station_from']) | set(edge_stats['station_to'])):,}")

# Stats par type de train
print(f"\n=== LIAISONS PAR TYPE DE TRAIN ===")
type_edge_counts = edge_stats_typed.groupby('train_type').agg(
    nb_liaisons=('min_time', 'count'),
    nb_trains_total=('nb_trains', 'sum')
).sort_values('nb_trains_total', ascending=False)
for t, row in type_edge_counts.iterrows():
    print(f"  {t:<20} {int(row['nb_liaisons']):>6} liaisons, {int(row['nb_trains_total']):>7} trains")

In [ ]:
def build_graph(edge_data):
    """Construit un graphe (liste d'adjacence) a partir d'un DataFrame d'aretes.
    graph[station_A] = [(station_B, temps_minutes, nb_trains, train_types), ...]
    """
    g = defaultdict(list)
    for _, row in edge_data.iterrows():
        types = row.get('train_types', [])
        if isinstance(types, str):
            types = [types]
        g[row['station_from']].append((
            row['station_to'],
            row['min_time'],
            row['nb_trains'],
            types if isinstance(types, list) else []
        ))
    return g


def graph_stats(g, name=""):
    """Affiche les stats d'un graphe."""
    all_st = set(g.keys())
    for neighbors in g.values():
        for sid, _, _, _ in neighbors:
            all_st.add(sid)
    degrees = [len(v) for v in g.values()] if g else [0]
    print(f"  Gares: {len(all_st):,}, Liaisons: {sum(len(v) for v in g.values()):,}, "
          f"Degre moyen: {np.mean(degrees):.1f}")


# ===== Graphe GLOBAL (tous les types de trains) =====
graph = build_graph(edge_stats)

# ===== Graphes par TYPE DE TRAIN =====
train_types_available = sorted(edge_stats_typed['train_type'].unique())
graphs_by_type = {}

for tt in train_types_available:
    subset = edge_stats_typed[edge_stats_typed['train_type'] == tt].copy()
    # Ajouter la colonne train_types pour compatibilite
    subset['train_types'] = [[tt]] * len(subset)
    graphs_by_type[tt] = build_graph(subset)

print(f"=== GRAPHES CONSTRUITS ===\n")
print(f"Graphe GLOBAL (tous trains) :")
graph_stats(graph)

print(f"\nGraphes par type de train :")
for tt in train_types_available:
    print(f"  [{tt}]")
    graph_stats(graphs_by_type[tt], tt)

In [ ]:
# Top 20 des gares les plus connectees
station_connections = [(area_id_to_name.get(s, s), len(graph[s])) for s in graph]
station_connections.sort(key=lambda x: -x[1])

print("Top 20 des gares les plus connectees :")
print(f"{'Gare':<45} {'Connexions':>10}")
print("-" * 57)
for name, count in station_connections[:20]:
    print(f"{name:<45} {count:>10}")

# Top 15 des liaisons les plus frequentes (nb de trains)
top_edges = edge_stats.nlargest(15, 'nb_trains')

print("\n\nTop 15 des liaisons les plus frequentes :")
print(f"{'De':<25} {'Vers':<25} {'Trains':>7} {'Temps min':>10} {'Types'}")
print("-" * 95)
for _, row in top_edges.iterrows():
    name_a = area_id_to_name.get(row['station_from'], row['station_from'])
    name_b = area_id_to_name.get(row['station_to'], row['station_to'])
    types_str = ', '.join(row.get('train_types', []))
    print(f"{name_a[:25]:<25} {name_b[:25]:<25} {int(row['nb_trains']):>7} {row['min_time']:>8.0f} min  {types_str}")

### Visualisation du graphe ferroviaire

Le graphe complet est affiche en utilisant les coordonnees GPS des gares comme positions.
- **Taille des noeuds** = nombre de connexions (les hubs sont plus gros)
- **Couleur des aretes** : rouge = TGV, bleu = TER, gris = mixte/autre
- **Labels** : seulement les 20 gares les plus connectees

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

# ===== GRAPHE DU RESEAU COMPLET =====

# Construire le graphe networkx
G = nx.DiGraph()

# Ajouter les noeuds avec positions GPS (lon=x, lat=y)
all_graph_stations = set(graph.keys())
for neighbors in graph.values():
    for edge in neighbors:
        all_graph_stations.add(edge[0])

pos = {}
for sid in all_graph_stations:
    lat = area_id_to_lat.get(sid)
    lon = area_id_to_lon.get(sid)
    if lat and lon:
        pos[sid] = (lon, lat)  # x=longitude, y=latitude
        G.add_node(sid)

# Ajouter les aretes avec couleur par type de train
edge_colors = []
for sid_from, neighbors in graph.items():
    for edge in neighbors:
        sid_to = edge[0]
        train_types = edge[3] if len(edge) > 3 else []
        if sid_from in pos and sid_to in pos:
            G.add_edge(sid_from, sid_to)
            # Couleur par type dominant
            has_tgv = any("TGV" in t for t in train_types)
            has_ter = any("TER" in t for t in train_types)
            if has_tgv and not has_ter:
                edge_colors.append("#e63946")  # rouge TGV
            elif has_ter and not has_tgv:
                edge_colors.append("#457b9d")  # bleu TER
            elif has_tgv and has_ter:
                edge_colors.append("#6a4c93")  # violet mixte
            else:
                edge_colors.append("#adb5bd")  # gris autre

# Taille des noeuds proportionnelle au degre
degrees = dict(G.degree())
node_sizes = [max(3, degrees.get(n, 1) * 1.5) for n in G.nodes()]

# Labels : seulement les 20 gares les plus connectees
top_stations = sorted(G.nodes(), key=lambda n: degrees.get(n, 0), reverse=True)[:20]
labels = {s: area_id_to_name.get(s, '')[:20] for s in top_stations}

# --- Affichage ---
fig, ax = plt.subplots(1, 1, figsize=(14, 14))
ax.set_facecolor("#f8f9fa")
fig.patch.set_facecolor("#f8f9fa")

# Aretes
nx.draw_networkx_edges(G, pos, ax=ax, edge_color=edge_colors, alpha=0.3, width=0.5,
                       arrows=False, connectionstyle="arc3,rad=0.0")
# Noeuds
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=node_sizes, node_color="#264653",
                       alpha=0.6, linewidths=0)
# Noeuds des hubs (plus gros, colores)
hub_sizes = [degrees.get(n, 1) * 8 for n in top_stations]
nx.draw_networkx_nodes(G, pos, nodelist=top_stations, ax=ax, node_size=hub_sizes,
                       node_color="#e76f51", alpha=0.9, linewidths=0.5, edgecolors="white")
# Labels
nx.draw_networkx_labels(G, pos, labels=labels, ax=ax, font_size=7, font_weight="bold",
                        font_color="#1d3557")

# Legende
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color="#e63946", lw=2, label="TGV INOUI"),
    Line2D([0], [0], color="#457b9d", lw=2, label="TER"),
    Line2D([0], [0], color="#6a4c93", lw=2, label="TGV + TER"),
    Line2D([0], [0], color="#adb5bd", lw=2, label="Autre (Intercites, Navette...)"),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e76f51', markersize=10,
           label="Hub principal (top 20)"),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=9, framealpha=0.9)

ax.set_title("Graphe du reseau ferroviaire SNCF\n(positions GPS, ~3 500 gares, ~10 800 liaisons)",
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel("Longitude", fontsize=10)
ax.set_ylabel("Latitude", fontsize=10)
ax.tick_params(labelsize=8)
plt.tight_layout()
plt.show()

print(f"Noeuds affiches: {G.number_of_nodes():,}")
print(f"Aretes affichees: {G.number_of_edges():,}")

---
## Etape 4 : Algorithme de Dijkstra

### Principe de Dijkstra :
- On part d'un noeud source avec distance 0
- On explore toujours le noeud non-visite le plus proche (via un **min-heap**)
- On met a jour les distances de ses voisins
- Quand on atteint la destination, on a le plus court chemin

**Poids des aretes** = temps de trajet minimum en minutes (pas la distance geographique !)

### Complexite : O((V + E) log V)
Avec V = nombre de gares et E = nombre de liaisons

In [ ]:
TRANSFER_PENALTY = 10  # minutes de correspondance a chaque gare intermediaire

def dijkstra(graph, start, end, transfer_penalty=TRANSFER_PENALTY):
    """
    Algorithme de Dijkstra pour trouver le plus court chemin
    entre deux gares dans le graphe ferroviaire.
    
    A chaque gare intermediaire (ni depart, ni arrivee), on ajoute
    une penalite de correspondance pour simuler le temps d'attente
    en gare (changement de quai, attente du prochain train, etc.).
    
    Args:
        graph: dict {station_id: [(voisin_id, temps_minutes, ...), ...]}
        start: station_id de depart
        end: station_id d'arrivee
        transfer_penalty: temps de correspondance en minutes (defaut: 10)
    
    Returns:
        (chemin, temps_total) ou (None, inf) si pas de chemin
        chemin = liste de station_ids
    """
    # Min-heap : (distance, compteur_unique, station_id)
    # Le compteur evite les comparaisons de strings si distances egales
    counter = 0
    heap = [(0.0, counter, start)]
    dist = {start: 0.0}
    prev = {start: None}
    visited = set()
    
    while heap:
        d, _, u = heapq.heappop(heap)
        
        if u == end:
            # Reconstruire le chemin
            path = []
            current = u
            while current is not None:
                path.append(current)
                current = prev[current]
            path.reverse()
            return path, d
        
        if u in visited:
            continue
        visited.add(u)
        
        # Penalite de correspondance a chaque gare intermediaire
        # (pas au depart, car on est deja en gare)
        penalty = transfer_penalty if (u != start) else 0
        
        for edge in graph.get(u, []):
            v, weight = edge[0], edge[1]
            if v in visited:
                continue
            alt = d + weight + penalty
            if alt < dist.get(v, float('inf')):
                dist[v] = alt
                prev[v] = u
                counter += 1
                heapq.heappush(heap, (alt, counter, v))
    
    return None, float('inf')

print(f"Fonction dijkstra() definie avec penalite de correspondance = {TRANSFER_PENALTY} min par gare intermediaire.")

---
## Etape 5 : Recherche ville -> gare

L'utilisateur tape un nom de ville (ex: "Paris", "Lyon", "Nantes").
On doit trouver la/les gare(s) correspondante(s) dans le graphe.

In [ ]:
from difflib import SequenceMatcher

def normalize(s):
    """Normalise un nom pour la recherche."""
    return s.lower().strip()


# Pre-calculer le nombre de connexions par gare (pour trier par importance)
station_degree = {}
for station_id, neighbors in graph.items():
    station_degree[station_id] = station_degree.get(station_id, 0) + len(neighbors)
    for edge in neighbors:
        station_degree[edge[0]] = station_degree.get(edge[0], 0) + 1


def find_station(query, graph, area_id_to_name, top_n=5):
    """
    Trouve les gares correspondant a une requete textuelle.
    
    Strategie de recherche (par priorite) :
    1. Correspondance exacte sur le nom complet
    2. La requete correspond a un mot complet au debut du nom 
       (ex: "Lyon" matche "Lyon Part Dieu" mais pas "Lyon-Vaise-Gare-Routiere")
    3. Le nom contient la requete
    4. Similarite fuzzy
    
    En cas d'egalite, on prefere les gares les plus connectees (= gares principales).
    
    Returns: liste de (station_id, station_name, score)
    """
    query_norm = normalize(query)
    results = []
    
    # Toutes les gares dans le graphe
    graph_stations = set(graph.keys())
    for neighbors in graph.values():
        for edge in neighbors:
            graph_stations.add(edge[0])
    
    for station_id in graph_stations:
        name = area_id_to_name.get(station_id, '')
        if not name:
            continue
        name_norm = normalize(name)
        
        # Bonus de connectivite (0 a 0.04) pour departager les ex-aequo
        degree = station_degree.get(station_id, 0)
        connectivity_bonus = min(0.04, degree / 1000.0)
        
        # 1. Correspondance exacte
        if query_norm == name_norm:
            results.append((station_id, name, 1.0 + connectivity_bonus))
            continue
        
        # 2. Le nom commence par "query " ou "query-" (mot complet au debut)
        #    Ex: "marseille" -> "Marseille Saint-Charles" (bon)
        #    mais pas: "marseille" -> "Marseille-en-Beauvaisis" (query fait partie d'un nom compose)
        if name_norm.startswith(query_norm):
            # Verifier si c'est un mot complet (suivi d'un espace ou fin de chaine)
            rest = name_norm[len(query_norm):]
            if rest == '':
                results.append((station_id, name, 1.0 + connectivity_bonus))
            elif rest[0] == ' ':
                # "Lyon Part Dieu" -> query="lyon", rest=" Part Dieu"
                results.append((station_id, name, 0.93 + connectivity_bonus))
            elif rest[0] == '-':
                # "Lyon-Vaise" -> nom compose avec tiret, moins pertinent
                results.append((station_id, name, 0.80 + connectivity_bonus))
            else:
                results.append((station_id, name, 0.75 + connectivity_bonus))
            continue
        
        # 3. Le nom contient la requete (pas au debut)
        if query_norm in name_norm:
            results.append((station_id, name, 0.70 + connectivity_bonus))
            continue
        
        # 4. Similarite fuzzy (seulement si > 0.6)
        ratio = SequenceMatcher(None, query_norm, name_norm).ratio()
        if ratio > 0.6:
            results.append((station_id, name, ratio * 0.5 + connectivity_bonus))
    
    results.sort(key=lambda x: -x[2])
    return results[:top_n]


# Test de la recherche
for city in ["Amiens", "Lyon", "Nantes", "Marseille", "Bordeaux", "Strasbourg", "Toulouse", "Rennes", "Lille"]:
    matches = find_station(city, graph, area_id_to_name)
    print(f"\n'{city}' :")
    for sid, name, score in matches[:3]:
        print(f"  [{score:.3f}] {name} (connexions: {station_degree.get(sid, 0)})")

NameError: name 'graph' is not defined

In [ ]:
def format_duration(minutes):
    """Formate une duree en minutes en 'Xh Ymin'."""
    if minutes is None or np.isnan(minutes):
        return "?"
    h = int(minutes // 60)
    m = int(minutes % 60)
    if h > 0:
        return f"{h}h{m:02d}"
    return f"{m}min"


def haversine(lat1, lon1, lat2, lon2):
    """Calcule la distance en km entre deux points GPS (formule de Haversine)."""
    R = 6371.0  # rayon de la Terre en km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return R * 2 * np.arcsin(np.sqrt(a))


def find_route(city_from, city_to, graph, area_id_to_name, train_type=None):
    """
    Trouve le trajet optimal entre deux villes.
    
    Args:
        city_from: nom de la ville de depart
        city_to: nom de la ville d'arrivee
        graph: le graphe a utiliser
        area_id_to_name: mapping station_id -> nom
        train_type: None=tous, ou str comme 'TGV INOUI', 'TER', etc.
    
    Returns: dict avec chemin, temps, details des segments
    """
    # Si un type de train est specifie, utiliser le graphe filtre
    if train_type and train_type in graphs_by_type:
        active_graph = graphs_by_type[train_type]
    else:
        active_graph = graph
    
    matches_from = find_station(city_from, active_graph, area_id_to_name)
    matches_to = find_station(city_to, active_graph, area_id_to_name)
    
    if not matches_from:
        return {"error": f"Aucune gare trouvee pour '{city_from}'"}
    if not matches_to:
        return {"error": f"Aucune gare trouvee pour '{city_to}'"}
    
    # Filtrer : garder seulement les matchs avec un score >= 70% du meilleur
    # Cela evite de tester "Santes" quand on cherche "Nantes"
    min_score_from = matches_from[0][2] * 0.7
    min_score_to = matches_to[0][2] * 0.7
    candidates_from = [(sid, name, sc) for sid, name, sc in matches_from[:3] if sc >= min_score_from]
    candidates_to = [(sid, name, sc) for sid, name, sc in matches_to[:3] if sc >= min_score_to]
    
    best_result = None
    best_time = float('inf')
    
    # Tester les combinaisons de gares candidates
    for start_id, start_name, _ in candidates_from:
        for end_id, end_name, _ in candidates_to:
            if start_id == end_id:
                continue
            path, total_time = dijkstra(active_graph, start_id, end_id)
            if path and total_time < best_time:
                best_time = total_time
                best_result = {
                    "from": start_name,
                    "to": end_name,
                    "from_id": start_id,
                    "to_id": end_id,
                    "path": path,
                    "total_time_min": total_time,
                    "total_time_str": format_duration(total_time),
                    "nb_stops": len(path),
                    "nb_correspondances": len(path) - 2
                }
    
    if best_result is None:
        return {"error": f"Aucun trajet trouve entre '{city_from}' et '{city_to}'"}
    
    # Details des segments
    segments = []
    total_distance = 0.0
    path = best_result["path"]
    for i in range(len(path) - 1):
        a, b = path[i], path[i + 1]
        name_a = area_id_to_name.get(a, a)
        name_b = area_id_to_name.get(b, b)
        seg_time = None
        seg_types = []
        for edge in active_graph.get(a, []):
            if edge[0] == b:
                seg_time = edge[1]
                seg_types = edge[3] if len(edge) > 3 else []
                break
        # Calcul de la distance geographique entre les deux gares
        lat_a = area_id_to_lat.get(a)
        lon_a = area_id_to_lon.get(a)
        lat_b = area_id_to_lat.get(b)
        lon_b = area_id_to_lon.get(b)
        if lat_a and lon_a and lat_b and lon_b:
            seg_dist = haversine(lat_a, lon_a, lat_b, lon_b)
        else:
            seg_dist = None
        if seg_dist:
            total_distance += seg_dist
        segments.append({
            "from": name_a,
            "to": name_b,
            "time_min": seg_time,
            "time_str": format_duration(seg_time) if seg_time else "?",
            "distance_km": seg_dist,
            "train_types": seg_types
        })
    best_result["segments"] = segments
    best_result["total_distance_km"] = total_distance
    best_result["train_type_filter"] = train_type
    
    return best_result


def print_route(result):
    """Affiche un trajet de maniere lisible."""
    if "error" in result:
        print(f"ERREUR: {result['error']}")
        return
    
    total_dist = result.get('total_distance_km', 0)
    total_time = result.get('total_time_min', 0)
    avg_speed = (total_dist / (total_time / 60)) if total_time > 0 and total_dist > 0 else 0
    
    filter_str = result.get('train_type_filter')
    mode_str = f"  Mode              : {filter_str}" if filter_str else "  Mode              : Tous trains"
    
    print(f"\n{'='*70}")
    print(f"  {result['from']}  -->  {result['to']}")
    print(f"{'='*70}")
    print(f"{mode_str}")
    print(f"  Temps total       : {result['total_time_str']}")
    print(f"  Distance totale   : {total_dist:.1f} km")
    print(f"  Vitesse moyenne   : {avg_speed:.0f} km/h")
    print(f"  Nb gares          : {result['nb_stops']}")
    print(f"  Correspondances   : {max(0, result['nb_correspondances'])}")
    print(f"{'_'*70}")
    print(f"  Itineraire :")
    print()
    
    for i, seg in enumerate(result['segments']):
        dist = seg.get('distance_km')
        dist_str = f"{dist:.1f} km" if dist else "? km"
        types = seg.get('train_types', [])
        types_str = f"[{', '.join(types)}]" if types else ""
        if i == 0:
            print(f"  DEPART    {seg['from']}")
        print(f"     |")
        print(f"     | {seg['time_str']}  -  {dist_str}  {types_str}")
        print(f"     v")
        if i == len(result['segments']) - 1:
            print(f"  ARRIVEE   {seg['to']}")
        else:
            print(f"     *      {seg['to']}")
    
    print(f"{'='*70}\n")

---
## Etape 6 : Tests avec des trajets reels

In [ ]:
# Test 1 : Paris -> Lyon
t0 = time.time()
result = find_route("Paris", "Lyon", graph, area_id_to_name)
elapsed = time.time() - t0
print_route(result)
print(f"  (calcul en {elapsed*1000:.0f} ms)")

In [ ]:
# Test 2 : Nantes -> Marseille
t0 = time.time()
result = find_route("Nantes", "Marseille", graph, area_id_to_name)
elapsed = time.time() - t0
print_route(result)
print(f"  (calcul en {elapsed*1000:.0f} ms)")

In [ ]:
# Test 3 : Bordeaux -> Strasbourg
t0 = time.time()
result = find_route("Bordeaux", "Strasbourg", graph, area_id_to_name)
elapsed = time.time() - t0
print_route(result)
print(f"  (calcul en {elapsed*1000:.0f} ms)")

In [ ]:
# Test 4 : Lille -> Toulouse
t0 = time.time()
result = find_route("Lille", "Toulouse", graph, area_id_to_name)
elapsed = time.time() - t0
print_route(result)
print(f"  (calcul en {elapsed*1000:.0f} ms)")

In [ ]:
# Test 5 : Rennes -> Nice
t0 = time.time()
result = find_route("Nantes", "Strasbourg", graph, area_id_to_name)
elapsed = time.time() - t0
print_route(result)
print(f"  (calcul en {elapsed*1000:.0f} ms)")

---
## Comparaison TGV vs TER

On compare le meme trajet en utilisant uniquement le reseau TGV, uniquement le reseau TER, ou tous les trains.

In [ ]:
# ============================
# Comparaison TGV vs TER
# ============================

def compare_routes(city_from, city_to):
    """Compare le trajet entre deux villes en TGV, TER, et tous trains."""
    print(f"\n{'#'*70}")
    print(f"  COMPARAISON : {city_from} -> {city_to}")
    print(f"{'#'*70}")
    
    for mode_name, mode_filter in [("TOUS TRAINS", None), ("TGV INOUI", "TGV INOUI"), ("TER", "TER")]:
        print(f"\n{'>'*5} {mode_name} {'<'*5}")
        t0 = time.time()
        result = find_route(city_from, city_to, graph, area_id_to_name, train_type=mode_filter)
        elapsed = time.time() - t0
        print_route(result)
        print(f"  (calcul en {elapsed*1000:.0f} ms)")

# Test : Paris -> Lyon (trajet classique TGV)
compare_routes("Paris", "Lyon")

In [ ]:
# Comparaison Nantes -> Strasbourg
compare_routes("Nantes", "Strasbourg")

In [ ]:
# Comparaison Bordeaux -> Lille
compare_routes("Bordeaux", "Lille")

---
## Visualisation d'un trajet sur le graphe

On affiche le trajet trouve par Dijkstra sur le reseau :
- **Noeuds rouges** = gares du trajet
- **Aretes rouges epaisses** = segments du trajet
- **Noeuds gris transparents** = gares voisines (1 saut) pour donner du contexte
- Les **temps** et **distances** sont affiches sur chaque segment

In [ ]:
def plot_route(result, active_graph=None, title_suffix="", ax=None):
    """
    Visualise un trajet sur le graphe ferroviaire.
    
    Args:
        result: dict retourne par find_route()
        active_graph: le graphe utilise (defaut: graph global)
        title_suffix: texte ajoute au titre
        ax: matplotlib axes (si None, cree une nouvelle figure)
    """
    if "error" in result:
        print(f"ERREUR: {result['error']}")
        return
    
    if active_graph is None:
        active_graph = graph
    
    path = result["path"]
    standalone = ax is None
    
    if standalone:
        fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    
    ax.set_facecolor("#f8f9fa")
    
    # Collecter les gares du trajet + voisines (1 saut de contexte)
    path_set = set(path)
    context_stations = set()
    for sid in path:
        for edge in active_graph.get(sid, []):
            if edge[0] not in path_set:
                context_stations.add(edge[0])
    
    all_display = path_set | context_stations
    
    # Positions GPS
    display_pos = {}
    for sid in all_display:
        lat = area_id_to_lat.get(sid)
        lon = area_id_to_lon.get(sid)
        if lat and lon:
            display_pos[sid] = (lon, lat)
    
    # Construire le sous-graphe
    subG = nx.DiGraph()
    for sid in all_display:
        if sid in display_pos:
            subG.add_node(sid)
    
    # Aretes de contexte (gris)
    context_edges = []
    for sid in all_display:
        if sid not in display_pos:
            continue
        for edge in active_graph.get(sid, []):
            if edge[0] in display_pos:
                subG.add_edge(sid, edge[0])
                if not (sid in path_set and edge[0] in path_set):
                    context_edges.append((sid, edge[0]))
    
    # Aretes du trajet (rouge)
    path_edges = []
    for i in range(len(path) - 1):
        a, b = path[i], path[i + 1]
        if a in display_pos and b in display_pos:
            path_edges.append((a, b))
            if not subG.has_edge(a, b):
                subG.add_edge(a, b)
    
    # --- Dessin ---
    
    # Aretes de contexte
    if context_edges:
        nx.draw_networkx_edges(subG, display_pos, edgelist=context_edges, ax=ax,
                               edge_color="#dee2e6", alpha=0.4, width=0.8, arrows=False)
    
    # Aretes du trajet
    nx.draw_networkx_edges(subG, display_pos, edgelist=path_edges, ax=ax,
                           edge_color="#e63946", alpha=0.9, width=3.0, arrows=True,
                           arrowsize=15, connectionstyle="arc3,rad=0.05")
    
    # Noeuds de contexte
    ctx_in_pos = [s for s in context_stations if s in display_pos]
    if ctx_in_pos:
        nx.draw_networkx_nodes(subG, display_pos, nodelist=ctx_in_pos, ax=ax,
                               node_size=30, node_color="#adb5bd", alpha=0.4)
        ctx_labels = {s: area_id_to_name.get(s, '')[:15] for s in ctx_in_pos}
        nx.draw_networkx_labels(subG, display_pos, labels=ctx_labels, ax=ax,
                                font_size=5, font_color="#6c757d", alpha=0.5)
    
    # Noeuds du trajet
    path_in_pos = [s for s in path if s in display_pos]
    nx.draw_networkx_nodes(subG, display_pos, nodelist=path_in_pos, ax=ax,
                           node_size=200, node_color="#e63946", alpha=0.9,
                           linewidths=1.5, edgecolors="white")
    
    # Depart et arrivee plus gros
    if path[0] in display_pos:
        nx.draw_networkx_nodes(subG, display_pos, nodelist=[path[0]], ax=ax,
                               node_size=400, node_color="#2d6a4f", alpha=0.95,
                               linewidths=2, edgecolors="white")
    if path[-1] in display_pos:
        nx.draw_networkx_nodes(subG, display_pos, nodelist=[path[-1]], ax=ax,
                               node_size=400, node_color="#1d3557", alpha=0.95,
                               linewidths=2, edgecolors="white")
    
    # Labels du trajet
    path_labels = {}
    for i, sid in enumerate(path):
        name = area_id_to_name.get(sid, sid)[:25]
        if i == 0:
            name = f"DEPART\n{name}"
        elif i == len(path) - 1:
            name = f"ARRIVEE\n{name}"
        path_labels[sid] = name
    path_labels_in_pos = {k: v for k, v in path_labels.items() if k in display_pos}
    nx.draw_networkx_labels(subG, display_pos, labels=path_labels_in_pos, ax=ax,
                            font_size=7, font_weight="bold", font_color="#1d3557")
    
    # Annotations temps/distance sur les aretes du trajet
    for i, seg in enumerate(result.get("segments", [])):
        a, b = path[i], path[i + 1]
        if a in display_pos and b in display_pos:
            mid_x = (display_pos[a][0] + display_pos[b][0]) / 2
            mid_y = (display_pos[a][1] + display_pos[b][1]) / 2
            dist = seg.get("distance_km")
            dist_str = f"{dist:.0f}km" if dist else ""
            label = f"{seg['time_str']}\n{dist_str}"
            ax.annotate(label, (mid_x, mid_y), fontsize=6, ha='center', va='center',
                       bbox=dict(boxstyle='round,pad=0.2', facecolor='white', 
                                 edgecolor='#e63946', alpha=0.85))
    
    # Titre
    mode = result.get('train_type_filter', 'Tous trains') or 'Tous trains'
    title = f"{result['from']} -> {result['to']}\n{result['total_time_str']} | {result.get('total_distance_km', 0):.0f} km | {mode}"
    if title_suffix:
        title += f" {title_suffix}"
    ax.set_title(title, fontsize=11, fontweight='bold', pad=10)
    ax.tick_params(labelsize=7)
    
    if standalone:
        plt.tight_layout()
        plt.show()

print("Fonction plot_route() definie.")

In [ ]:
# Visualisation d'un trajet : Nantes -> Marseille
result_viz = find_route("Nantes", "Marseille", graph, area_id_to_name)
print_route(result_viz)
plot_route(result_viz)

### Comparaison visuelle TGV vs TER

Pour un meme trajet, on affiche cote a cote :
- **A gauche** : le trajet en TGV (peu d'arrets, aretes longues)
- **A droite** : le trajet en TER (beaucoup d'arrets, aretes courtes)

Meme echelle geographique pour bien voir la difference.

In [ ]:
def compare_routes_visual(city_from, city_to):
    """Compare visuellement un trajet en TGV vs TER cote a cote."""
    
    result_tgv = find_route(city_from, city_to, graph, area_id_to_name, train_type="TGV INOUI")
    result_ter = find_route(city_from, city_to, graph, area_id_to_name, train_type="TER")
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    fig.patch.set_facecolor("#f8f9fa")
    
    # Calculer les limites communes (meme echelle pour les deux)
    all_stations = set()
    for r in [result_tgv, result_ter]:
        if "error" not in r:
            all_stations.update(r["path"])
            for sid in r["path"]:
                g_to_use = graphs_by_type.get(r.get("train_type_filter", ""), graph)
                for edge in g_to_use.get(sid, []):
                    all_stations.add(edge[0])
    
    lats = [area_id_to_lat.get(s) for s in all_stations if area_id_to_lat.get(s)]
    lons = [area_id_to_lon.get(s) for s in all_stations if area_id_to_lon.get(s)]
    
    if lats and lons:
        margin = 0.5
        xlim = (min(lons) - margin, max(lons) + margin)
        ylim = (min(lats) - margin, max(lats) + margin)
    else:
        xlim, ylim = None, None
    
    # TGV
    if "error" not in result_tgv:
        plot_route(result_tgv, active_graph=graphs_by_type.get("TGV INOUI", graph), ax=axes[0])
    else:
        axes[0].text(0.5, 0.5, f"Pas de trajet TGV\n{city_from} -> {city_to}",
                     ha='center', va='center', fontsize=14, transform=axes[0].transAxes)
        axes[0].set_title(f"TGV INOUI : {city_from} -> {city_to}", fontsize=11, fontweight='bold')
    
    # TER
    if "error" not in result_ter:
        plot_route(result_ter, active_graph=graphs_by_type.get("TER", graph), ax=axes[1])
    else:
        axes[1].text(0.5, 0.5, f"Pas de trajet TER\n{city_from} -> {city_to}",
                     ha='center', va='center', fontsize=14, transform=axes[1].transAxes)
        axes[1].set_title(f"TER : {city_from} -> {city_to}", fontsize=11, fontweight='bold')
    
    # Appliquer les memes limites
    if xlim and ylim:
        for ax in axes:
            ax.set_xlim(xlim)
            ax.set_ylim(ylim)
    
    fig.suptitle(f"Comparaison visuelle : {city_from} -> {city_to}",
                 fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    
    # Resume textuel
    print(f"\n{'='*60}")
    print(f"  RESUME : {city_from} -> {city_to}")
    print(f"{'='*60}")
    print(f"  {'':20s} {'TGV INOUI':>15s} {'TER':>15s}")
    print(f"  {'-'*50}")
    
    tgv_time = result_tgv.get('total_time_str', '-') if 'error' not in result_tgv else "Pas de trajet"
    ter_time = result_ter.get('total_time_str', '-') if 'error' not in result_ter else "Pas de trajet"
    tgv_dist = f"{result_tgv.get('total_distance_km', 0):.0f} km" if 'error' not in result_tgv else "-"
    ter_dist = f"{result_ter.get('total_distance_km', 0):.0f} km" if 'error' not in result_ter else "-"
    tgv_stops = str(result_tgv.get('nb_stops', '-')) if 'error' not in result_tgv else "-"
    ter_stops = str(result_ter.get('nb_stops', '-')) if 'error' not in result_ter else "-"
    
    print(f"  {'Temps total':<20s} {tgv_time:>15s} {ter_time:>15s}")
    print(f"  {'Distance':<20s} {tgv_dist:>15s} {ter_dist:>15s}")
    print(f"  {'Nb gares':<20s} {tgv_stops:>15s} {ter_stops:>15s}")
    print(f"{'='*60}")

print("Fonction compare_routes_visual() definie.")

In [ ]:
# Comparaison visuelle Paris -> Lyon (TGV vs TER)
compare_routes_visual("Paris", "Lyon")

In [ ]:
# Comparaison visuelle Nantes -> Strasbourg (TGV vs TER)
compare_routes_visual("Nantes", "Strasbourg")

In [ ]:
# ============================
# Fonction interactive
# ============================

def recherche_trajet():
    """Interface interactive pour rechercher un trajet."""
    print("\n" + "="*50)
    print("  RECHERCHE DE TRAJET SNCF")
    print("="*50)
    
    city_from = input("\nVille de depart : ").strip()
    city_to = input("Ville d'arrivee : ").strip()
    
    if not city_from or not city_to:
        print("Veuillez entrer les deux villes.")
        return
    
    print(f"\nRecherche : {city_from} -> {city_to}...")
    
    t0 = time.time()
    result = find_route(city_from, city_to, graph, area_id_to_name)
    elapsed = time.time() - t0
    
    print_route(result)
    print(f"  (calcul en {elapsed*1000:.0f} ms)")

# Decommenter pour utiliser :
# recherche_trajet()

---
## Resume

### Ce qu'on a fait :
1. **Charge les donnees GTFS** SNCF (gares, trajets, horaires)
2. **Construit un graphe ferroviaire** reel a partir des `stop_times` (aretes = liaisons directes entre gares)
3. **Poids = temps de trajet** minimum en minutes (pas la distance geographique)
4. **Dijkstra** trouve le chemin le plus rapide entre deux gares
5. **Recherche fuzzy** pour matcher un nom de ville vers une gare du reseau

### Difference avec l'ancien `pathfinding.py` :
| | Ancien (geo.csv) | Nouveau (GTFS) |
|---|---|---|
| Source | geo.csv (departements/villes) | SNCF stop_times + stops |
| Graphe | K plus proches voisins (geographique) | Connexions reelles (trains) |
| Poids | Distance a vol d'oiseau (km) | Temps de trajet reel (minutes) |
| Pertinence | Faible (pas de lien avec le reseau) | Forte (basee sur les vrais trajets) |